# Deterministic Actor-Critic: DPG, DDPG, and TD3

>- **NAME:** **REDACTED**
>- **STUDENT ID:** **REDACTED**

In this notebook, the goal is to build deterministic actor-critic methods from the ground up and connect each implementation detail to the theory from the homework. We will move from the deterministic policy-gradient identity to practical deep continuous-control algorithms, and then study why DDPG can fail and how TD3 fixes several of its failure modes.

We will implement and compare the following methods:

$$
\text{DPG}
\quad\longrightarrow\quad
\text{DDPG}
\quad\longrightarrow\quad
\text{TD3}.
$$

This cell standardizes the PyTorch installation used by the notebook. It is needed if you want to use P100 on Kaggle!

First, it removes any preinstalled versions of torch, torchvision, and torchaudio. Kaggle environments often come with these packages already installed, but their versions may differ from the versions expected by this homework. The uninstall command is allowed to fail without stopping the notebook, because some of the packages may not be installed yet.

Second, it installs fixed versions of the PyTorch stack:

- torch==2.7.1
- torchvision==0.22.1
- torchaudio==2.7.1

Using exact versions makes the notebook more reproducible across students and avoids version mismatches between PyTorch, torchvision, and torchaudio.

In [ ]:
# !python -m pip uninstall -y torch torchvision torchaudio || true

!python -m pip install --no-cache-dir --force-reinstall torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu126

This cell installs Gymnasium with MuJoCo support.

In [ ]:
!pip install "gymnasium[mujoco]"

## Import Necessary Libraries

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from moving_target_reacher_env import make_moving_reacher, rollout_to_video, embed_video

Note: Using a GPU is not strictly necessary for this assignment. The neural networks used here are small, and the majority of training time is spent running episodes in the environment (which occurs on the CPU), rather than performing backpropagation updates.

## Explore the Environment

The environment used in this project is a modified version of the Gymnasium MuJoCo `Reacher-v5` task. In the original Reacher environment, a two-link robotic arm must move its fingertip toward a target location. Here, the task is extended into a moving-target tracking problem, where the target position changes over time according to a predefined trajectory.

The environment is implemented as a wrapper around the base Reacher environment. At every timestep, the wrapper updates the target position using a target trajectory function and then recomputes the reward based on the current distance between the fingertip and the moving target. This changes the task from simply reaching a fixed point into continuously tracking a dynamic target.

Three target motion types are supported. A static target keeps the target fixed at a specified position. A circular target moves the target along a circular path using a radius, angular velocity, and center point. A Lissajous target produces a more complex periodic trajectory using separate amplitudes and angular velocities for the x and y directions. These different motion types allow the agent to be evaluated under both simple and more challenging tracking conditions.

The observation can include the original Reacher observation, the current target position, the current target velocity, time, and a phase representation based on sine and cosine features. Including the target position and velocity gives the agent direct information about where the target is and how it is moving, while the phase features provide a compact representation of periodic motion.

The action space is kept the same as the original Reacher environment. The agent outputs continuous torque commands for the arm joints, and these actions are applied to the MuJoCo simulation. The goal of the policy is to choose actions that keep the fingertip close to the moving target throughout the episode.

The reward consists of two components: a distance penalty and a control penalty. The distance penalty encourages the fingertip to stay close to the target, while the control penalty discourages unnecessarily large actions. Therefore, the agent must learn a balance between accurate tracking and smooth, efficient control.

At the start of each episode, the environment time and step counter are reset to zero, and the target is placed at its initial position. During each environment step, the target is synchronized with the current simulation time, the action is applied, the time is advanced, and the target is moved to its next position. Episodes terminate or truncate according to the underlying environment and the configured maximum episode length.

### Task Description

In this task, you will create and inspect a moving-target Reacher environment. The environment uses the MuJoCo `Reacher-v5` task with a circularly moving target. The target follows a circular trajectory with radius `0.15` and angular velocity `1.0`, and each episode is limited to `100` steps.

The observation includes the target position, target velocity, and phase information, allowing the agent to observe both where the target is and how it is moving. To verify that the environment works correctly, you will reset the environment, take random actions from the action space for a short rollout, and print useful diagnostic information such as the reward, fingertip-target distance, and target position at each step.

In [ ]:
# Create the circular moving-target Reacher environment
env = make_moving_reacher(
    motion="circle",
    radius=0.15,
    omega=1.0,
    max_episode_steps=100,
    include_target_pos=True,
    include_target_vel=True,
    include_phase=True,
)

# Reset the environment
obs, info = env.reset(seed=0)

print("Observation shape:", obs.shape)
print("Action space:", env.action_space)
print("Initial target position:", (info["target_x"], info["target_y"]))

# Initialize total reward
total_reward = 0.0

# Run a 10-step random-action rollout
for step in range(10):
    action = env.action_space.sample()

    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward

    print(
        f"Step {step + 1:02d} | "
        f"Reward: {reward:8.4f} | "
        f"Distance: {info['distance']:.4f} | "
        f"Target: ({info['target_x']:.4f}, {info['target_y']:.4f})"
    )

    if terminated or truncated:
        print("Episode ended early.")
        break

# Print total reward
print(f"\nTotal reward over the test: {total_reward:.4f}")

# Close the environment
env.close()

### Task Description

In this task, you will generate a visualization of a rollout in the moving-target Reacher environment. The environment uses a circular target trajectory with radius `0.15`, angular velocity `1.0`, and an episode length of `150` steps. The observation includes the target position, target velocity, and phase features.

Instead of using the MuJoCo renderer, the rollout is recorded using a custom no-render video recorder. The recorder extracts the Reacher state directly from the environment and draws each frame with Matplotlib, including the robotic arm, moving target, fingertip position, target trail, fingertip trail, reward statistics, distance information, and action values.

A random policy is used by setting `policy=None`, so actions are sampled directly from the environment action space. The rollout is saved as a video file, summary statistics are printed, and the resulting video is embedded in the notebook for inspection.

In [ ]:
# Create the videos directory
Path("videos").mkdir(parents=True, exist_ok=True)

# Create the circular moving-target Reacher environment
env = make_moving_reacher(
    motion="circle",
    radius=0.15,
    omega=1.0,
    max_episode_steps=150,
    include_target_pos=True,
    include_target_vel=True,
    include_phase=True,
)

try:
    # Record a random-policy rollout
    video_path, rollout_stats = rollout_to_video(
        env=env,
        policy=None,  # Random actions
        video_path="videos/random_policy_rollout.mp4",
        max_steps=150,
        fps=30,
        seed=0,
        deterministic=True,
        record_every=1,
        trail_len=20,
    )
finally:
    env.close()

# Print rollout statistics
print("Saved video:", video_path)
print("Rollout statistics:")
for key, value in rollout_stats.items():
    print(f"  {key}: {value}")

# Display the generated video
display(embed_video(video_path, width=700))

## DPG

### Task Description

Implement a minimal Deterministic Policy Gradient agent for the moving-target Reacher environment. The agent should use:

- an actor network for deterministic continuous actions
- a critic network for estimating \(Q(s, a)\)
- Gaussian exploration noise
- optional target networks
- one-step TD critic updates
- actor updates through the critic

The actor outputs bounded actions using:

$$
a = b + s \tanh(f_\theta(o))
$$

where:

$$
s = \frac{a_{\text{high}} - a_{\text{low}}}{2},
\qquad
b = \frac{a_{\text{high}} + a_{\text{low}}}{2}
$$

The critic target is:

$$
y = r + \gamma (1 - d) Q(o', \mu(o'))
$$

or, when target networks are enabled:

$$
y = r + \gamma (1 - d) Q_{\text{targ}}(o', \mu_{\text{targ}}(o'))
$$

The critic minimizes:

$$
L_Q = \left(Q(o, a) - y\right)^2
$$

The actor maximizes the critic value, implemented as minimizing:

$$
L_\mu = -Q(o, \mu(o))
$$

For target networks, use soft updates:

$$
\theta_{\text{targ}} \leftarrow (1-\tau)\theta_{\text{targ}} + \tau\theta
$$

In [ ]:
# -----------------------------
# Utilities
# -----------------------------

def mlp(sizes, activation=nn.ReLU, output_activation=nn.Identity):
    layers = []

    for i in range(len(sizes) - 1):
        act = activation if i < len(sizes) - 2 else output_activation
        layers += [
            nn.Linear(sizes[i], sizes[i + 1]),
            act(),
        ]

    return nn.Sequential(*layers)


@torch.no_grad()
def soft_update(net, target_net, tau):
    for parameter, target_parameter in zip(
        net.parameters(), target_net.parameters()
    ):
        target_parameter.mul_(1.0 - tau)
        target_parameter.add_(parameter, alpha=tau)


@torch.no_grad()
def hard_update(net, target_net):
    target_net.load_state_dict(net.state_dict())


# -----------------------------
# Actor and critic
# -----------------------------

class DeterministicActor(nn.Module):
    def __init__(self, obs_dim, act_dim, act_low, act_high,
        hidden=(256, 256),
    ):
        super().__init__()

        self.net = mlp(
            [obs_dim, *hidden, act_dim],
            activation=nn.ReLU,
            output_activation=nn.Identity,
        )

        act_low = torch.as_tensor(act_low, dtype=torch.float32)
        act_high = torch.as_tensor(act_high, dtype=torch.float32)

        self.register_buffer(
            "act_scale",
            (act_high - act_low) / 2.0,
        )
        self.register_buffer(
            "act_bias",
            (act_high + act_low) / 2.0,
        )

    def forward(self, obs):
        raw_action = self.net(obs)

        bounded_action = (
            self.act_bias
            + self.act_scale * torch.tanh(raw_action)
        )

        return bounded_action


class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=(256, 256)):
        super().__init__()

        self.q = mlp(
            [obs_dim + act_dim, *hidden, 1],
            activation=nn.ReLU,
            output_activation=nn.Identity,
        )

    def forward(self, obs, act):
        obs_act = torch.cat([obs, act], dim=-1)
        q_value = self.q(obs_act)

        return q_value.squeeze(-1)


# -----------------------------
# Exploration noise
# -----------------------------

class GaussianNoise:
    def __init__(self, act_dim, std=0.1):
        self.act_dim = act_dim
        self.std = std

    def sample(self):
        return np.random.normal(
            loc=0.0,
            scale=self.std,
            size=self.act_dim,
        ).astype(np.float32)

    def reset(self):
        # Independent Gaussian noise has no internal state.
        pass


# -----------------------------
# DPG config
# -----------------------------

@dataclass
class DPGConfig:
    gamma: float = 0.99
    actor_lr: float = 1e-4
    critic_lr: float = 1e-3
    tau: float = 0.005
    noise_std: float = 0.1
    hidden: tuple = (256, 256)
    use_target_networks: bool = False


# -----------------------------
# DPG agent
# -----------------------------

class DPGAgent:
    def __init__(
        self,
        env,
        cfg=DPGConfig(),
        device=None,
    ):
        self.cfg = cfg

        self.device = torch.device(
            device
            or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.obs_dim = int(np.prod(env.observation_space.shape))
        self.act_dim = int(np.prod(env.action_space.shape))

        self.act_low = env.action_space.low.astype(np.float32)
        self.act_high = env.action_space.high.astype(np.float32)

        self.actor = DeterministicActor(
            self.obs_dim,
            self.act_dim,
            self.act_low,
            self.act_high,
            cfg.hidden,
        ).to(self.device)

        self.critic = Critic(
            self.obs_dim,
            self.act_dim,
            cfg.hidden,
        ).to(self.device)

        self.actor_targ = DeterministicActor(
            self.obs_dim,
            self.act_dim,
            self.act_low,
            self.act_high,
            cfg.hidden,
        ).to(self.device)

        self.critic_targ = Critic(
            self.obs_dim,
            self.act_dim,
            cfg.hidden,
        ).to(self.device)

        hard_update(self.actor, self.actor_targ)
        hard_update(self.critic, self.critic_targ)

        self.actor_opt = torch.optim.Adam(
            self.actor.parameters(),
            lr=cfg.actor_lr,
        )

        self.critic_opt = torch.optim.Adam(
            self.critic.parameters(),
            lr=cfg.critic_lr,
        )

        self.noise = GaussianNoise(
            self.act_dim,
            cfg.noise_std,
        )

        # Minimal DPG stores only the most recent transition.
        self.last_transition = None

    @torch.no_grad()
    def select_action(self, obs, explore=True):
        obs_tensor = torch.as_tensor(
            obs,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        action = self.actor(obs_tensor)
        action = action.squeeze(0).cpu().numpy()

        if explore:
            action = action + self.noise.sample()

        action = np.clip(
            action,
            self.act_low,
            self.act_high,
        )

        return action.astype(np.float32)

    def store(
        self,
        obs,
        act,
        rew,
        next_obs,
        done,
    ):
        self.last_transition = (
            np.asarray(obs, dtype=np.float32).copy(),
            np.asarray(act, dtype=np.float32).copy(),
            float(rew),
            np.asarray(next_obs, dtype=np.float32).copy(),
            float(done),
        )

    def update(self, step=None):
        if self.last_transition is None:
            return {}

        obs, act, rew, next_obs, done = (
            self.last_transition
        )

        obs_tensor = torch.as_tensor(
            obs,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        act_tensor = torch.as_tensor(
            act,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        rew_tensor = torch.as_tensor(
            [rew],
            dtype=torch.float32,
            device=self.device,
        )

        next_obs_tensor = torch.as_tensor(
            next_obs,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        done_tensor = torch.as_tensor(
            [done],
            dtype=torch.float32,
            device=self.device,
        )

        with torch.no_grad():
            if self.cfg.use_target_networks:
                next_action = self.actor_targ(
                    next_obs_tensor
                )
                next_q = self.critic_targ(
                    next_obs_tensor,
                    next_action,
                )
            else:
                next_action = self.actor(
                    next_obs_tensor
                )
                next_q = self.critic(
                    next_obs_tensor,
                    next_action,
                )

            target_q = (
                rew_tensor
                + self.cfg.gamma
                * (1.0 - done_tensor)
                * next_q
            )

        current_q = self.critic(
            obs_tensor,
            act_tensor,
        )

        critic_loss = F.mse_loss(
            current_q,
            target_q,
        )

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            self.critic.parameters(),
            max_norm=10.0,
        )

        self.critic_opt.step()

        for parameter in self.critic.parameters():
            parameter.requires_grad_(False)

        policy_action = self.actor(obs_tensor)

        actor_loss = -self.critic(
            obs_tensor,
            policy_action,
        ).mean()

        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.actor.parameters(),
            max_norm=10.0,
        )

        self.actor_opt.step()

        for parameter in self.critic.parameters():
            parameter.requires_grad_(True)

        if self.cfg.use_target_networks:
            soft_update(
                self.actor,
                self.actor_targ,
                self.cfg.tau,
            )
            soft_update(
                self.critic,
                self.critic_targ,
                self.cfg.tau,
            )

        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "q": float(current_q.mean().item()),
        }

### Task Description

Implement reusable training and evaluation utilities for the DPG agent. These functions will also be used for the next methods, so keep the interface general: the training loop should work with any agent that implements `select_action`, `store`, `update`, and `noise.reset`, while the evaluation loop should run the policy without exploration noise.

During training, the agent interacts with the environment, stores the latest transition, updates its networks, logs episode returns and lengths, and resets the environment at episode boundaries.

For bootstrapping, use only true termination as the terminal signal:

$$
d_{\text{target}} = \mathbb{1}[\text{terminated}]
$$

Do not treat time-limit truncation as terminal for the TD target.

Evaluation should run deterministic actions:

$$
a = \mu(o)
$$

and report the mean return, return standard deviation, and mean fingertip-target distance when available.

In [ ]:
def train_agent(
    env,
    agent,
    steps=50000,
    seed=0,
    log_every=1000,
):
    obs, info = env.reset(seed=seed)

    ep_ret = 0.0
    ep_len = 0
    ep = 0

    history = {
        "returns": [],
        "lengths": [],
        "losses": [],
        "evals": [],
    }

    for t in range(1, steps + 1):
        # Select an exploratory action
        action = agent.select_action(obs, explore=True)

        # Interact with the environment
        next_obs, rew, terminated, truncated, info = env.step(action)
        done_for_target = float(terminated)

        # Store the transition and update the agent
        agent.store(
            obs,
            action,
            rew,
            next_obs,
            done_for_target,
        )
        losses = agent.update(step=t)

        obs = next_obs
        ep_ret += float(rew)
        ep_len += 1

        if losses:
            history["losses"].append(
                {
                    "step": t,
                    **losses,
                }
            )

        if terminated or truncated:
            history["returns"].append(ep_ret)
            history["lengths"].append(ep_len)

            ep += 1
            obs, info = env.reset(seed=seed + ep)

            agent.noise.reset()

            ep_ret = 0.0
            ep_len = 0

        if t % log_every == 0:
            recent = history["returns"][-10:]
            avg_ret = np.mean(recent) if recent else np.nan

            print(
                f"step={t:6d} | "
                f"episodes={ep:4d} | "
                f"recent_return={avg_ret:8.2f}"
            )

    return history

In [ ]:
def evaluate(env, agent, episodes=5, seed=10000):
    returns = []
    distances = []

    for ep in range(episodes):
        agent.noise.reset()
        obs, info = env.reset(seed=seed + ep)

        done = False
        ep_ret = 0.0
        ep_dist = []

        while not done:
            action = agent.select_action(obs, explore=False)

            next_obs, reward, terminated, truncated, info = env.step(action)

            ep_ret += float(reward)

            done = terminated or truncated

            if "distance" in info:
                ep_dist.append(float(info["distance"]))

            obs = next_obs

        # Store total episode return
        returns.append(ep_ret)

        if ep_dist:
            distances.append(float(np.mean(ep_dist)))

    out = {
        "eval_return": float(np.mean(returns)),
        "eval_return_std": float(np.std(returns)),
    }

    if distances:
        out["eval_distance"] = float(np.mean(distances))

    return out

### Train and Evaluate Vanilla DPG

In [ ]:
np.random.seed(0)
torch.manual_seed(0)

cfg = DPGConfig(
    gamma=0.99,
    actor_lr=1e-4,
    critic_lr=1e-3,
    noise_std=0.1,
    hidden=(256, 256),
    use_target_networks=False,
)

agent = DPGAgent(
    env=env,
    cfg=cfg,
)

print("Training device:", agent.device)

# Train for 25,000 environment steps
dpg_history = train_agent(
    env=env,
    agent=agent,
    steps=25_000,
    seed=0,
    log_every=1_000,
)

In [ ]:
eval_stats = evaluate(
    env,
    agent,
    episodes=10,
    seed=10_000,
)

print("DPG evaluation:")
for key, value in eval_stats.items():
    print(f"  {key}: {value:.4f}")

video_dir = Path("videos")
video_dir.mkdir(parents=True, exist_ok=True)

def learned_policy(obs):
    return agent.select_action(obs, explore=False)

video_path, video_stats = rollout_to_video(
    env=env,
    policy=learned_policy,
    video_path=video_dir / "dpg_learned_policy.mp4",
    max_steps=150,
    fps=30,
    seed=20_000,
    trail_len=20,
)

print("\nDPG video rollout:")
for key, value in video_stats.items():
    print(f"  {key}: {value}")

display(embed_video(video_path, width=720))

## DDPG

### Task Description

Extend the previous DPG agent into DDPG by adding:

- replay-buffer training
- target actor and target critic networks
- Ornstein-Uhlenbeck exploration noise
- minibatch updates instead of single-transition updates

The same `train_agent` and `evaluate` functions from the previous section should be used here as well.

OU exploration noise:

$$
x_{t+1}
=
x_t
+
\theta(\mu - x_t)\Delta t
+
\sigma \sqrt{\Delta t}\epsilon_t,
\qquad
\epsilon_t \sim \mathcal{N}(0, I)
$$

DDPG critic target:

$$
y
=
r
+
\gamma(1-d)
Q_{\text{targ}}(o', \mu_{\text{targ}}(o'))
$$

Critic loss:

$$
L_Q
=
\frac{1}{B}
\sum_i
\left(Q(o_i, a_i) - y_i\right)^2
$$

Actor loss:

$$
L_\mu
=
-\frac{1}{B}
\sum_i
Q(o_i, \mu(o_i))
$$

Target-network update:

$$
\theta_{\text{targ}}
\leftarrow
(1-\tau)\theta_{\text{targ}} + \tau\theta
$$

In [ ]:
# -----------------------------
# Replay Buffer
# -----------------------------

class ReplayBuffer:
    def __init__(self, obs_dim, act_dim, size):
        self.obs_buf = np.zeros((size, obs_dim), dtype=np.float32)
        self.next_obs_buf = np.zeros((size, obs_dim), dtype=np.float32)
        self.act_buf = np.zeros((size, act_dim), dtype=np.float32)
        self.rew_buf = np.zeros(size, dtype=np.float32)
        self.done_buf = np.zeros(size, dtype=np.float32)

        self.ptr = 0
        self.size = 0
        self.max_size = size

    def store(self, obs, act, rew, next_obs, done):
        self.obs_buf[self.ptr] = np.asarray(
            obs,
            dtype=np.float32,
        )
        self.act_buf[self.ptr] = np.asarray(
            act,
            dtype=np.float32,
        )
        self.rew_buf[self.ptr] = float(rew)
        self.next_obs_buf[self.ptr] = np.asarray(
            next_obs,
            dtype=np.float32,
        )
        self.done_buf[self.ptr] = float(done)

        # Circular-buffer pointer
        self.ptr = (self.ptr + 1) % self.max_size

        # Current number of valid transitions
        self.size = min(
            self.size + 1,
            self.max_size,
        )

    def sample_batch(self, batch_size, device="cpu"):
        if self.size < batch_size:
            raise ValueError(
                f"Replay buffer has {self.size} samples, "
                f"but batch_size={batch_size}."
            )

        # Uniform random sampling from valid transitions
        idxs = np.random.randint(
            low=0,
            high=self.size,
            size=batch_size,
        )

        batch = {
            "obs": self.obs_buf[idxs],
            "act": self.act_buf[idxs],
            "rew": self.rew_buf[idxs],
            "next_obs": self.next_obs_buf[idxs],
            "done": self.done_buf[idxs],
        }

        return {
            key: torch.as_tensor(
                value,
                dtype=torch.float32,
                device=device,
            )
            for key, value in batch.items()
        }


# -----------------------------
# Ornstein-Uhlenbeck noise
# -----------------------------

class OUNoise:
    def __init__(self, act_dim, std=0.1, theta=0.15, dt=0.02, mu=0.0):
        self.act_dim = act_dim
        self.std = std
        self.theta = theta
        self.dt = dt
        self.mu = mu

        self.state = np.ones(self.act_dim, dtype=np.float32) * self.mu

    def reset(self):
        self.state.fill(self.mu)

    def sample(self):
        epsilon = np.random.randn(
            self.act_dim
        ).astype(np.float32)

        dx = (
            self.theta
            * (self.mu - self.state)
            * self.dt
            + self.std
            * np.sqrt(self.dt)
            * epsilon
        )

        self.state = self.state + dx

        return self.state.copy().astype(np.float32)


# -----------------------------
# DDPG config
# -----------------------------

@dataclass
class DDPGConfig(DPGConfig):
    replay_size: int = 100_000
    batch_size: int = 128
    update_after: int = 1_000
    update_every: int = 1

    use_target_networks: bool = True
    use_ou_noise: bool = True


# -----------------------------
# DDPG agent
# -----------------------------

class DDPGAgent(DPGAgent):
    def __init__(self, env, cfg=DDPGConfig(), device=None):
        super().__init__(env, cfg, device)

        # TODO: create replay buffer
        self.replay = ReplayBuffer(
            obs_dim=self.obs_dim,
            act_dim=self.act_dim,
            size=cfg.replay_size,
        )

        # Replace independent Gaussian exploration with
        # temporally correlated OU noise.
        if cfg.use_ou_noise:
            self.noise = OUNoise(
                act_dim=self.act_dim,
                std=cfg.noise_std,
                theta=0.15,
                dt=getattr(env, "dt", 0.02),
                mu=0.0,
            )

        self.total_stored = 0

    def store(self, obs, act, rew, next_obs, done):

        self.replay.store(
            obs=obs,
            act=act,
            rew=rew,
            next_obs=next_obs,
            done=done,
        )

        self.total_stored += 1

    def update(self, step=None):
        minimum_size = max(
            self.cfg.update_after,
            self.cfg.batch_size,
        )

        if self.replay.size < minimum_size:
            return {}

        if (
            step is not None
            and step % self.cfg.update_every != 0
        ):
            return {}

        batch = self.replay.sample_batch(
            batch_size=self.cfg.batch_size,
            device=self.device,
        )

        obs = batch["obs"]
        act = batch["act"]
        rew = batch["rew"]
        next_obs = batch["next_obs"]
        done = batch["done"]

        # -------------------------
        # Critic target
        # -------------------------

        with torch.no_grad():
            next_act = self.actor_targ(next_obs)

            next_q = self.critic_targ(
                next_obs,
                next_act,
            )

            target_q = (
                rew
                + self.cfg.gamma
                * (1.0 - done)
                * next_q
            )

        # -------------------------
        # Critic update
        # -------------------------

        q = self.critic(obs, act)

        critic_loss = F.mse_loss(
            q,
            target_q,
        )

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.critic.parameters(),
            max_norm=10.0,
        )

        self.critic_opt.step()

        # -------------------------
        # Actor update
        # -------------------------

        for parameter in self.critic.parameters():
            parameter.requires_grad_(False)

        policy_act = self.actor(obs)

        actor_loss = -self.critic(
            obs,
            policy_act,
        ).mean()

        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.actor.parameters(),
            max_norm=10.0,
        )

        self.actor_opt.step()

        for parameter in self.critic.parameters():
            parameter.requires_grad_(True)

        # -------------------------
        # Target-network update
        # -------------------------

        soft_update(
            self.actor,
            self.actor_targ,
            self.cfg.tau,
        )
        soft_update(
            self.critic,
            self.critic_targ,
            self.cfg.tau,
        )

        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "q": float(q.mean().item()),
            "target_q": float(target_q.mean().item()),
            "replay_size": self.replay.size,
        }

### Train and Evaluate DDPG Agent

In [ ]:
# -----------------------------
# DDPG configuration
# -----------------------------

# Reproducibility
np.random.seed(1)
torch.manual_seed(1)

ddpg_cfg = DDPGConfig(
    gamma=0.99,               # discount factor
    actor_lr=1e-4,            # actor learning rate
    critic_lr=1e-3,           # critic learning rate
    tau=0.005,                # soft target-network update rate
    noise_std=0.2,            # OU exploration noise scale
    hidden=(256, 256),        # hidden-layer sizes
    replay_size=100_000,      # replay buffer capacity
    batch_size=128,           # minibatch size
    update_after=1_000,       # begin learning after 1,000 transitions
    update_every=1,           # one update per environment step
    use_target_networks=True,
    use_ou_noise=True,
)


# -----------------------------
# Create and train DDPG agent
# -----------------------------

ddpg_agent = DDPGAgent(
    env=env,
    cfg=ddpg_cfg,
)

print("DDPG training device:", ddpg_agent.device)

ddpg_history = train_agent(
    env=env,
    agent=ddpg_agent,
    steps=25_000,
    seed=1,
    log_every=1_000,
)

In [ ]:
# -----------------------------
# Evaluate the trained agent
# -----------------------------

eval_stats = evaluate(
    env,
    ddpg_agent,
    episodes=10,
    seed=30_000,
)

print("\nDDPG evaluation:")
for key, value in eval_stats.items():
    print(f"  {key}: {value:.4f}")


# -----------------------------
# Deterministic video policy
# -----------------------------

def ddpg_policy(obs):
    return ddpg_agent.select_action(
        obs,
        explore=False,
    )


# Ensure the output directory exists
video_dir = Path("videos")
video_dir.mkdir(parents=True, exist_ok=True)


# Record a rollout video using the learned policy
video_path, video_stats = rollout_to_video(
    env=env,
    policy=ddpg_policy,
    video_path=video_dir / "ddpg_learned_policy.mp4",
    max_steps=150,
    fps=30,
    seed=40_000,
    trail_len=20,
)

print("\nDDPG video rollout:")
for key, value in video_stats.items():
    print(f"  {key}: {value}")

display(embed_video(video_path, width=720))

## TD3

### Task Description

Implement TD3 by extending the previous DDPG agent. TD3 modifies DDPG using three main changes:

1. **Clipped double Q-learning**: use two critics and bootstrap from the smaller target value.
2. **Target policy smoothing**: add clipped noise to the target action.
3. **Delayed policy updates**: update the actor and target networks less frequently than the critics.

Target action:

$$
a' = \text{clip}\left(\mu_{\text{targ}}(o') + \epsilon,\ a_{\min},\ a_{\max}\right)
$$

where:

$$
\epsilon \sim \text{clip}\left(\mathcal{N}(0, \sigma^2), -c, c\right)
$$

TD3 target:

$$
y = r + \gamma(1-d)\min_{i=1,2} Q_{i,\text{targ}}(o', a')
$$

Critic loss:

$$
L_Q =
\text{MSE}(Q_1(o,a), y)
+
\text{MSE}(Q_2(o,a), y)
$$

Delayed actor loss:

$$
L_\mu = -\frac{1}{B}\sum_i Q_1(o_i, \mu(o_i))
$$

Use the same replay buffer, training loop, and evaluation function from the previous DDPG section.

In [ ]:
# -----------------------------
# TD3 config
# -----------------------------

@dataclass
class TD3Config(DDPGConfig):
    policy_delay: int = 2
    target_noise_std: float = 0.2
    target_noise_clip: float = 0.5


# -----------------------------
# TD3 agent
# -----------------------------

class TD3Agent(DDPGAgent):
    def __init__(
        self,
        env,
        cfg=TD3Config(),
        device=None,
    ):
        super().__init__(env, cfg, device)

        self.critic1 = self.critic
        self.critic1_targ = self.critic_targ

        self.critic2 = Critic(
            obs_dim=self.obs_dim,
            act_dim=self.act_dim,
            hidden=cfg.hidden,
        ).to(self.device)

        self.critic2_targ = Critic(
            obs_dim=self.obs_dim,
            act_dim=self.act_dim,
            hidden=cfg.hidden,
        ).to(self.device)

        hard_update(
            self.critic2,
            self.critic2_targ,
        )

        self.critic_opt = torch.optim.Adam(
            list(self.critic1.parameters())
            + list(self.critic2.parameters()),
            lr=cfg.critic_lr,
        )

        self.total_updates = 0

    def update(self, step=None):
        minimum_size = max(
            self.cfg.update_after,
            self.cfg.batch_size,
        )

        if self.replay.size < minimum_size:
            return {}

        if (
            step is not None
            and step % self.cfg.update_every != 0
        ):
            return {}

        self.total_updates += 1

        batch = self.replay.sample_batch(
            batch_size=self.cfg.batch_size,
            device=self.device,
        )

        obs = batch["obs"]
        act = batch["act"]
        rew = batch["rew"]
        next_obs = batch["next_obs"]
        done = batch["done"]

        # --------------------------------------------------
        # TD3 critic target
        # --------------------------------------------------

        with torch.no_grad():
            # Sample Gaussian target-policy noise.
            noise = torch.randn_like(act)
            noise = noise * self.cfg.target_noise_std

            # Clip the target-policy noise.
            noise = torch.clamp(
                noise,
                -self.cfg.target_noise_clip,
                self.cfg.target_noise_clip,
            )

            # Smooth the target action.
            next_act = self.actor_targ(next_obs) + noise

            # Convert environment action limits to tensors.
            act_low = torch.as_tensor(
                self.act_low,
                dtype=torch.float32,
                device=self.device,
            )

            act_high = torch.as_tensor(
                self.act_high,
                dtype=torch.float32,
                device=self.device,
            )

            # Clip the perturbed action to valid action bounds.
            next_act = torch.max(
                torch.min(next_act, act_high),
                act_low,
            )

            # Evaluate both target critics using the same
            # smoothed target action.
            q1_targ = self.critic1_targ(
                next_obs,
                next_act,
            )

            q2_targ = self.critic2_targ(
                next_obs,
                next_act,
            )

            # Clipped double Q-learning.
            q_targ_min = torch.minimum(
                q1_targ,
                q2_targ,
            )

            target_q = (
                rew
                + self.cfg.gamma
                * (1.0 - done)
                * q_targ_min
            )

        # --------------------------------------------------
        # Critic update
        # --------------------------------------------------

        q1 = self.critic1(obs, act)
        q2 = self.critic2(obs, act)

        critic1_loss = F.mse_loss(
            q1,
            target_q,
        )

        critic2_loss = F.mse_loss(
            q2,
            target_q,
        )

        critic_loss = critic1_loss + critic2_loss

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(self.critic1.parameters())
            + list(self.critic2.parameters()),
            max_norm=10.0,
        )

        self.critic_opt.step()

        metrics = {
            "critic_loss": float(critic_loss.item()),
            "critic1_loss": float(critic1_loss.item()),
            "critic2_loss": float(critic2_loss.item()),
            "q1": float(q1.mean().item()),
            "q2": float(q2.mean().item()),
            "target_q": float(target_q.mean().item()),
            "replay_size": self.replay.size,
        }

        # --------------------------------------------------
        # Delayed actor and target-network updates
        # --------------------------------------------------

        if self.total_updates % self.cfg.policy_delay == 0:
            # Freeze both critics during the actor update.
            for parameter in self.critic1.parameters():
                parameter.requires_grad_(False)

            for parameter in self.critic2.parameters():
                parameter.requires_grad_(False)

            # TD3 updates the actor using Q1 only.
            policy_action = self.actor(obs)

            actor_loss = -self.critic1(
                obs,
                policy_action,
            ).mean()

            self.actor_opt.zero_grad(set_to_none=True)
            actor_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.actor.parameters(),
                max_norm=10.0,
            )

            self.actor_opt.step()

            for parameter in self.critic1.parameters():
                parameter.requires_grad_(True)

            for parameter in self.critic2.parameters():
                parameter.requires_grad_(True)

            # Target networks are updated only when the actor
            # receives its delayed update.
            soft_update(
                self.actor,
                self.actor_targ,
                self.cfg.tau,
            )

            soft_update(
                self.critic1,
                self.critic1_targ,
                self.cfg.tau,
            )

            soft_update(
                self.critic2,
                self.critic2_targ,
                self.cfg.tau,
            )

            metrics["actor_loss"] = float(
                actor_loss.item()
            )

        return metrics

### Train and Evaluate TD3 Agent

In [ ]:
# -----------------------------
# TD3 configuration
# -----------------------------

np.random.seed(2)
torch.manual_seed(2)

td3_cfg = TD3Config(
    gamma=0.99,                  # discount factor
    actor_lr=1e-4,               # actor learning rate
    critic_lr=1e-3,              # critic learning rate
    tau=0.005,                   # target-network update rate

    noise_std=0.2,

    replay_size=100_000,
    batch_size=128,
    update_after=1_000,
    update_every=1,

    # TD3 additions
    policy_delay=2,
    target_noise_std=0.2,
    target_noise_clip=0.5,

    hidden=(256, 256),

    use_target_networks=True,
    use_ou_noise=True,
)


# -----------------------------
# Create and train TD3 agent
# -----------------------------

td3_agent = TD3Agent(
    env=env,
    cfg=td3_cfg,
)

print("TD3 training device:", td3_agent.device)

td3_history = train_agent(
    env=env,
    agent=td3_agent,
    steps=25_000,
    seed=2,
    log_every=1_000,
)

In [ ]:
# -----------------------------
# Evaluate the trained agent
# -----------------------------

eval_stats = evaluate(
    env,
    td3_agent,
    episodes=10,
    seed=50_000,
)

print("\nTD3 evaluation:")
for key, value in eval_stats.items():
    print(f"  {key}: {value:.4f}")


# -----------------------------
# Deterministic video policy
# -----------------------------

def td3_policy(obs):
    return td3_agent.select_action(
        obs,
        explore=False,
    )


video_dir = Path("videos")
video_dir.mkdir(parents=True, exist_ok=True)


# Record a rollout video using the TD3 policy
video_path, video_stats = rollout_to_video(
    env=env,
    policy=td3_policy,
    video_path=video_dir / "td3_learned_policy.mp4",
    max_steps=150,
    fps=30,
    seed=60_000,
    trail_len=20,
)

print("\nTD3 video rollout:")
for key, value in video_stats.items():
    print(f"  {key}: {value}")

display(embed_video(video_path, width=720))

### Ablations

In this final section, you will study the effect of each TD3 improvement by training and evaluating several TD3-like agents.

Create three ablated TD3 agents. Each agent should remove exactly one TD3 addition while keeping the rest of the algorithm unchanged:

1. **No clipped double-Q learning**  
   Use only one critic, as in DDPG.

2. **No target policy smoothing**  
   Do not add clipped noise to the target action.

3. **No delayed policy updates**  
   Update the actor at every critic update instead of every `policy_delay` steps.

Train each ablated agent using the same environment, training loop, number of steps, and evaluation protocol as the full TD3 agent. Then compare their learning curves against:

- DPG
- DDPG
- full TD3
- the three TD3 ablations

Use the results to discuss which TD3 component has the largest effect on performance and stability in the moving-target Reacher task.

In [ ]:
class TD3NoDoubleQAgent(DDPGAgent):
    def __init__(self, env, cfg=TD3Config(), device=None):
        super().__init__(env, cfg, device)
        self.total_updates = 0

    def update(self, step=None):
        minimum_size = max(
            self.cfg.update_after,
            self.cfg.batch_size,
        )

        if self.replay.size < minimum_size:
            return {}

        if (
            step is not None
            and step % self.cfg.update_every != 0
        ):
            return {}

        self.total_updates += 1

        batch = self.replay.sample_batch(
            batch_size=self.cfg.batch_size,
            device=self.device,
        )

        obs = batch["obs"]
        act = batch["act"]
        rew = batch["rew"]
        next_obs = batch["next_obs"]
        done = batch["done"]

        # --------------------------------------------------
        # Single-critic target with target policy smoothing
        # --------------------------------------------------
        with torch.no_grad():
            base_next_act = self.actor_targ(next_obs)

            noise = (
                torch.randn_like(base_next_act)
                * self.cfg.target_noise_std
            )

            noise = torch.clamp(
                noise,
                -self.cfg.target_noise_clip,
                self.cfg.target_noise_clip,
            )

            next_act = base_next_act + noise

            act_low = torch.as_tensor(
                self.act_low,
                dtype=torch.float32,
                device=self.device,
            )

            act_high = torch.as_tensor(
                self.act_high,
                dtype=torch.float32,
                device=self.device,
            )

            next_act = torch.max(
                torch.min(next_act, act_high),
                act_low,
            )

            next_q = self.critic_targ(
                next_obs,
                next_act,
            )

            target_q = (
                rew
                + self.cfg.gamma
                * (1.0 - done)
                * next_q
            )

        # --------------------------------------------------
        # Critic update
        # --------------------------------------------------
        q = self.critic(obs, act)

        critic_loss = F.mse_loss(
            q,
            target_q,
        )

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.critic.parameters(),
            max_norm=10.0,
        )

        self.critic_opt.step()

        metrics = {
            "critic_loss": float(critic_loss.item()),
            "q": float(q.mean().item()),
            "target_q": float(target_q.mean().item()),
            "replay_size": self.replay.size,
        }

        # --------------------------------------------------
        # Delayed actor and target updates
        # --------------------------------------------------
        if self.total_updates % self.cfg.policy_delay == 0:
            for parameter in self.critic.parameters():
                parameter.requires_grad_(False)

            policy_action = self.actor(obs)

            actor_loss = -self.critic(
                obs,
                policy_action,
            ).mean()

            self.actor_opt.zero_grad(set_to_none=True)
            actor_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                self.actor.parameters(),
                max_norm=10.0,
            )

            self.actor_opt.step()

            for parameter in self.critic.parameters():
                parameter.requires_grad_(True)

            soft_update(
                self.actor,
                self.actor_targ,
                self.cfg.tau,
            )

            soft_update(
                self.critic,
                self.critic_targ,
                self.cfg.tau,
            )

            metrics["actor_loss"] = float(
                actor_loss.item()
            )

        return metrics

In [ ]:
from dataclasses import replace

ABLATION_STEPS = 25_000
ABLATION_SEED = 2

def set_experiment_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_ablation(agent_class, config, name):
    print("\n" + "=" * 70)
    print(f"Training: {name}")
    print("=" * 70)

    # Use the same initialization and environment seed for fairness.
    set_experiment_seed(ABLATION_SEED)

    ablation_agent = agent_class(
        env=env,
        cfg=config,
    )

    ablation_history = train_agent(
        env=env,
        agent=ablation_agent,
        steps=ABLATION_STEPS,
        seed=ABLATION_SEED,
        log_every=1_000,
    )

    return ablation_agent, ablation_history


# 1
no_double_q_cfg = replace(td3_cfg)

no_double_q_agent, no_double_q_history = train_ablation(
    agent_class=TD3NoDoubleQAgent,
    config=no_double_q_cfg,
    name="TD3 without clipped double-Q",
)


# 2
no_smoothing_cfg = replace(
    td3_cfg,
    target_noise_std=0.0,
    target_noise_clip=0.0,
)

no_smoothing_agent, no_smoothing_history = train_ablation(
    agent_class=TD3Agent,
    config=no_smoothing_cfg,
    name="TD3 without target policy smoothing",
)

# 3
no_delay_cfg = replace(
    td3_cfg,
    policy_delay=1,
)

no_delay_agent, no_delay_history = train_ablation(
    agent_class=TD3Agent,
    config=no_delay_cfg,
    name="TD3 without delayed policy updates",
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


EVAL_EPISODES = 10
EVAL_SEED = 70_000


agents = {
    "DPG": agent,
    "DDPG": ddpg_agent,
    "TD3": td3_agent,
    "TD3: no double-Q": no_double_q_agent,
    "TD3: no smoothing": no_smoothing_agent,
    "TD3: no delay": no_delay_agent,
}

histories = {
    "DPG": dpg_history,
    "DDPG": ddpg_history,
    "TD3": td3_history,
    "TD3: no double-Q": no_double_q_history,
    "TD3: no smoothing": no_smoothing_history,
    "TD3: no delay": no_delay_history,
}


evaluation_results = {}

for name, current_agent in agents.items():
    stats = evaluate(
        env=env,
        agent=current_agent,
        episodes=EVAL_EPISODES,
        seed=EVAL_SEED,
    )

    evaluation_results[name] = stats

    print(
        f"{name:22s} | "
        f"return={stats['eval_return']:8.3f} ± "
        f"{stats['eval_return_std']:7.3f} | "
        f"distance={stats.get('eval_distance', np.nan):7.4f}"
    )


def final_episode_statistics(history, last_n=20):
    episode_returns = np.asarray(
        history["returns"],
        dtype=np.float32,
    )

    if episode_returns.size == 0:
        return np.nan, np.nan

    tail = episode_returns[-last_n:]

    return (
        float(np.mean(tail)),
        float(np.std(tail)),
    )


comparison_rows = []

for name in agents:
    tail_mean, tail_std = final_episode_statistics(
        histories[name],
        last_n=20,
    )

    result = evaluation_results[name]

    comparison_rows.append(
        {
            "Agent": name,
            "Evaluation return": result["eval_return"],
            "Evaluation return std": result["eval_return_std"],
            "Evaluation distance": result.get(
                "eval_distance",
                np.nan,
            ),
            "Last-20 training return": tail_mean,
            "Last-20 training std": tail_std,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "Evaluation return",
    ascending=False,
).reset_index(drop=True)

display(comparison_df)

In [ ]:
def smoothed_learning_curve(history, window=10):
    returns = np.asarray(
        history["returns"],
        dtype=np.float32,
    )

    lengths = np.asarray(
        history["lengths"],
        dtype=np.int64,
    )

    episode_steps = np.cumsum(lengths)

    if len(returns) == 0:
        return np.array([]), np.array([])

    if len(returns) < window:
        return episode_steps, returns

    kernel = np.ones(window, dtype=np.float32) / window

    smoothed_returns = np.convolve(
        returns,
        kernel,
        mode="valid",
    )

    smoothed_steps = episode_steps[window - 1:]

    return smoothed_steps, smoothed_returns


plt.figure(figsize=(12, 7))

for name, history in histories.items():
    curve_steps, curve_returns = smoothed_learning_curve(
        history,
        window=10,
    )

    plt.plot(
        curve_steps,
        curve_returns,
        label=name,
        linewidth=2,
    )

plt.xlabel("Environment steps")
plt.ylabel("Episode return — 10-episode moving average")
plt.title("DPG, DDPG, TD3, and TD3 ablations")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()